In [ ]:
import scanpy as sc
import numpy as np
import pandas as pd
import plotnine as gg
from tqdm import tqdm
import seaborn as sns
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import pandas as pd
import scipy.stats as stats
from scipy.spatial.distance import squareform
from scipy.cluster.hierarchy import linkage, fcluster
from fba_utils import (
    plot_similarity_matrix,
    hamming_distance,
    compute_pairwise,
    to_long_no_diagonal,
    get_gene_topology_stats,
)
from essential.fba import load_ecoli_rich_medium_model

from essential.data import load_fitness_data

pd.set_option("display.max_columns", 500)

SHARED_THEME = gg.theme(
    axis_text=gg.element_text(size=6),
    axis_title=gg.element_text(size=7),
    figure_size=(3, 2),
    title=gg.element_text(size=7),
    legend_text=gg.element_text(size=6),
)

In [ ]:
ecoli_model = load_ecoli_rich_medium_model()

In [ ]:
fitness_df = load_fitness_data()
fitness_df_gene = fitness_df.groupby("gene")[["T1", "T2", "T3", "T4"]].mean()

flux_df = pd.read_csv("/workspace/experiments/01232026_fba/data/moma_fluxes.csv", index_col=0)
worker_df = pd.read_csv("/workspace/experiments/01232026_fba/data/worker_ids.csv", index_col=0)
wt_flux = pd.read_csv("/workspace/experiments/01232026_fba/data/wt_fluxes_0.csv", index_col=0)
growth_df_raw = pd.read_csv(
    "/workspace/experiments/01232026_fba/data/fba_growth_ratios.csv", index_col=0
)

flux_df_bin = (flux_df.abs() >= 1e-6).astype(float)
wt_flux_bin = (wt_flux.abs() >= 1e-6).astype(float).T

flux_ham_dist = hamming_distance(flux_df_bin)
flux_ham_dist_df = pd.DataFrame(flux_ham_dist, index=flux_df_bin.index, columns=flux_df_bin.index)
d_to_wt = hamming_distance(flux_df_bin, wt_flux_bin).squeeze()

growth_df_raw.loc[flux_df_bin.index, "d_to_wt"] = d_to_wt

topology_stats_df = get_gene_topology_stats(ecoli_model, growth_df_raw.index)

growth_df = (
    growth_df_raw.merge(fitness_df_gene, left_index=True, right_index=True)
    .merge(topology_stats_df, left_index=True, right_index=True)
    .assign(
        fba_growth_type=lambda x: pd.Categorical(
            np.where(x["growth_ratio"] >= 0.5, "high", "low"), categories=["low", "high"]
        ),
        growth_score=lambda x: (x["growth"] - x["growth_wt"]) / x["growth_wt"],
        is_predicted_essential=lambda x: x["growth_ratio"] < 0.5,
    )
    .merge(worker_df, left_index=True, right_index=True)
)

In [ ]:
adata = sc.read_h5ad(
    "/workspace/data/251117_genomescale_CRISPRi/sample_mix_umi200_hvg500_pc25_neighbors10_mindist0.55.scvi.h5ad"
)
adata_case = sc.read_h5ad("/workspace/data/251117_genomescale_CRISPRi/adata_case.annotated.h5ad")

transcript_df = []
transcript_case_df = []
z_transcript_df = []
z_transcript_case_df = []
gene_names = []
gene_case_names = []


for gene in tqdm(adata.obs["gene"].unique()):
    adata_gene = adata[adata.obs["gene"] == gene]
    X_gene = adata_gene.layers["cp10k"].toarray()
    if X_gene.shape[0] > 0:
        gene_names.append(gene)
        transcript_df.append(X_gene.mean(axis=0))
        z_transcript_df.append(adata_gene.obsm["X_scVI"].mean(axis=0))

    adata_gene_case = adata_case[adata_case.obs["gene"] == gene]
    X_gene_case = adata_gene_case.layers["cp10k"].toarray()
    if X_gene_case.shape[0] > 0:
        gene_case_names.append(gene)
        transcript_case_df.append(X_gene_case.mean(axis=0))
        z_transcript_case_df.append(adata_gene_case.obsm["X_scVI"].mean(axis=0))
transcript_df = pd.DataFrame(transcript_df, index=gene_names)
transcript_case_df = pd.DataFrame(transcript_case_df, index=gene_case_names)
z_transcript_df = pd.DataFrame(z_transcript_df, index=gene_names)
z_transcript_case_df = pd.DataFrame(z_transcript_case_df, index=gene_case_names)

In [ ]:
# transcript_df_pca = PCA(n_components=50).fit_transform(transcript_df)
# transcript_df_pca_ = pd.DataFrame(transcript_df_pca, index=gene_names)
# transcript_pairwise = compute_pairwise(transcript_df_pca_, metric="euclidean")

# transcript_df_pca = PCA(n_components=50).fit_transform(transcript_case_df)
# transcript_df_pca_ = pd.DataFrame(transcript_df_pca, index=gene_case_names)
# transcript_pairwise = compute_pairwise(transcript_df_pca_, metric="euclidean")

transcript_pairwise = compute_pairwise(z_transcript_case_df, metric="euclidean")

In [ ]:
genes_to_remove = [
    "coaA",
    "coaD",
    "coaE",
    "dfp",
    "folB",
    "folC",
    "folE",
    "folK",
    "folP",
    "nadA",
    "nadB",
    "nadC",
    "nadD",
    "nadE",
    "nadK",
    "ribA",
    "ribB",
    "ribC",
    "ribD",
    "ribE",
    "ribF",
    "bioA",
    "bioB",
    "bioC",
    "bioD",
    "bioF",
    "bioH",
    "dxr",
    "dxs",
    "ispA",
    "ispB",
    "ispD",
    "ispE",
    "ispF",
    "ispG",
    "ispH",
    "ispU",
    "moaA",
    "moaB",
    "moaC",
    "moaD",
    "moaE",
    "moeA",
    "moeB",
    "mog",
    "mobA",
    "ubiA",
    "ubiC",
    "ubiD",
    "ubiX",
    "pdxA",
    "pdxB",
    "pdxJ",
    "thiL",
    "iscS",
    "cyaY",
    "cysG",
    "metK",
    "pabA",
    "pabB",
    "pabC",
]

pred_essential_genes = growth_df[growth_df["is_predicted_essential"] == True].index
core_metabolic_set = np.setdiff1d(pred_essential_genes, genes_to_remove)

In [ ]:
essential_genes = growth_df.loc[growth_df["growth_ratio"] < 0.5].index.tolist()
flux_ham_dist_df_essential = flux_ham_dist_df.loc[essential_genes, essential_genes]

# Exploration

#### Gene nearest neighbors in flux space are at various distances

In [ ]:
flux_ham_dist_df_long = (
    flux_ham_dist_df.stack()
    .reset_index()
    .rename(columns={"level_0": "gene1", "level_1": "gene2", 0: "hamming_distance"})
)

min_hamming_dist = (
    flux_ham_dist_df_long.loc[lambda x: x["gene1"] != x["gene2"]]
    .groupby("gene1")["hamming_distance"]
    .min()
    .to_frame("min_hamming")
    .merge(growth_df, left_index=True, right_index=True)
)

fig = (
    gg.ggplot(min_hamming_dist, gg.aes(x="min_hamming", color="is_predicted_essential"))
    + gg.stat_ecdf()
    + gg.theme_minimal()
    + gg.labs(x="Minimum Hamming distance", y="Cumulative Density")
    + SHARED_THEME
    + gg.theme(legend_position="bottom", figure_size=(3, 2))
)
fig.save("min_hamming_dist_ecdf.png", dpi=500, bbox_inches="tight")
display(fig)

fig2 = (
    gg.ggplot(min_hamming_dist, gg.aes(x="min_hamming"))
    + gg.geom_histogram(bins=100)
    + gg.theme_minimal()
    + gg.labs(x="Minimum Hamming distance", y="Cumulative Density", title="All genes")
    + SHARED_THEME
    + gg.theme(legend_position="bottom", figure_size=(3, 2))
)
fig2.save("min_hamming_dist_hist.png", dpi=500, bbox_inches="tight")
display(fig2)

fig3 = (
    gg.ggplot(min_hamming_dist.loc[core_metabolic_set], gg.aes(x="min_hamming"))
    + gg.geom_histogram(bins=100)
    + gg.theme_minimal()
    + gg.labs(x="Minimum Hamming distance", y="Cumulative Density", title="Core metabolic set")
    + SHARED_THEME
    + gg.theme(legend_position="bottom", figure_size=(3, 2))
)
fig3.save("min_hamming_dist_hist_core_metabolic_set.png", dpi=500, bbox_inches="tight")
display(fig3)

#### (validation) gene knockdows embedded by MOMA flux decompose in two groups, one essential and one non-essential

In [ ]:
g = plot_similarity_matrix(
    flux_ham_dist_df,
    growth_df,
    row_color_column="growth_ratio",
    similarity_label="Hamming distance",
)
g.savefig("./flux_clustermap_all_hamming.png", dpi=500, bbox_inches="tight")
plt.show()

#### Focus on essential genes

In [ ]:
flux_ham_dist_df_essential

In [ ]:
essential_gene_metadata = growth_df.loc[essential_genes]
essential_gene_metadata["cofactor biosynthesis"] = essential_gene_metadata.index.isin(
    genes_to_remove
)
essential_gene_metadata["cofactor biosynthesis"] = np.where(
    essential_gene_metadata["cofactor biosynthesis"], "cofactor biosynthesis", "other"
)
essential_gene_metadata["cofactor biosynthesis"].unique()

In [ ]:
condensed_dist = squareform(flux_ham_dist_df_essential.values)
Z = linkage(condensed_dist, method="complete")

In [ ]:
unique_clusters = sorted(essential_gene_metadata["cofactor biosynthesis"].unique())
palette = sns.color_palette("tab10", n_colors=len(unique_clusters))
cluster_color_map = dict(zip(unique_clusters, palette))

row_colors = essential_gene_metadata["cofactor biosynthesis"].map(cluster_color_map)

g = sns.clustermap(
    flux_ham_dist_df_essential,
    row_linkage=Z,
    col_linkage=Z,
    row_colors=row_colors,
    col_colors=row_colors,
    cmap="rocket_r",
    xticklabels=False,
    yticklabels=False,
    cbar_pos=(0.02, 0.8, 0.05, 0.18),
    cbar_kws={"label": "Hamming Distance"},
)

legend_patches = [
    mpatches.Patch(color=color, label=f"Cluster {cluster_id}")
    for cluster_id, color in cluster_color_map.items()
]

# Place the legend slightly outside the heatmap
g.ax_heatmap.legend(
    handles=legend_patches,
    title="Clusters",
    loc="center left",
    bbox_to_anchor=(1.02, 0.5),
    frameon=False,
)

plt.savefig("./flux_clustermap_essential_cofactor_biosynthesis.png", dpi=500, bbox_inches="tight")
plt.show()

In [ ]:
pred_essential_genes = growth_df[growth_df["is_predicted_essential"] == True].index
flux_dist_selected = flux_ham_dist_df.loc[pred_essential_genes, pred_essential_genes]
growth_df_selected = growth_df.loc[pred_essential_genes].copy()
growth_df_selected["num_downstream_reactions_"] = np.clip(
    growth_df_selected["num_downstream_reactions"], 0, 50
)

plot_similarity_matrix(
    flux_dist_selected,
    growth_df_selected,
    "num_reactions",
)
plt.show()

In [ ]:
pd.set_option("display.max_rows", 500)

In [ ]:
neg_genes = ["coaA", "folC", "ribE", "nadC", "tmk"]
pos_genes = ["hemE", "hemA", "gltX"]

In [ ]:
display(growth_df_selected.loc[neg_genes, ["num_reactions", "num_downstream_reactions", "d_to_wt"]])
display(growth_df_selected.loc[pos_genes, ["num_reactions", "num_downstream_reactions", "d_to_wt"]])

In [ ]:
# def find_differential_reactions(a, b):
#     assert a.index.equals(b.index)
#     disagrees = a != b
#     return a.index[disagrees]

# flux_df_bin = (flux_df.abs() >= 1e-6).astype(float)
# wt_flux_bin = (wt_flux.abs() >= 1e-6).astype(float).squeeze()

# for gene in neg_genes:
#     print(gene)
#     print(", ".join(find_differential_reactions(flux_df_bin.loc[gene], wt_flux_bin)))
#     print()

# print()
# for gene in pos_genes:
#     print(gene)
#     print(", ".join(find_differential_reactions(flux_df_bin.loc[gene], wt_flux_bin)))
#     print()

In [ ]:
condensed_dist = squareform(flux_ham_dist_df_essential.values)
Z = linkage(condensed_dist, method="complete")
distance_threshold = 30.0
clusters_dist = fcluster(Z, t=distance_threshold, criterion="distance")
cluster_assignments = pd.Series(clusters_dist, index=flux_ham_dist_df_essential.index)

In [ ]:
unique_clusters = sorted(cluster_assignments.unique())
palette = sns.color_palette("tab10", n_colors=len(unique_clusters))
cluster_color_map = dict(zip(unique_clusters, palette))

row_colors = cluster_assignments.map(cluster_color_map)

g = sns.clustermap(
    flux_ham_dist_df_essential,
    row_linkage=Z,
    col_linkage=Z,
    row_colors=row_colors,
    col_colors=row_colors,
    cmap="rocket_r",
    xticklabels=False,
    yticklabels=False,
    cbar_pos=(0.02, 0.8, 0.05, 0.18),
    cbar_kws={"label": "Hamming Distance"},
)

legend_patches = [
    mpatches.Patch(color=color, label=f"Cluster {cluster_id}")
    for cluster_id, color in cluster_color_map.items()
]

# Place the legend slightly outside the heatmap
g.ax_heatmap.legend(
    handles=legend_patches,
    title="Clusters",
    loc="center left",
    bbox_to_anchor=(1.02, 0.5),
    frameon=False,
)

plt.show()

In [ ]:
# print genes in each cluster
for cluster_id in range(1, len(cluster_assignments.unique() + 1)):
    gene_list = cluster_assignments[cluster_assignments == cluster_id].index.tolist()
    print(f"Cluster {cluster_id}: {gene_list}")

#### Hard way: directly remove pathological genes

In [ ]:
flux_dist_selected = flux_ham_dist_df.loc[core_metabolic_set, core_metabolic_set]
growth_df_selected = growth_df.loc[core_metabolic_set].copy()

core_metabolic_set_ = np.intersect1d(core_metabolic_set, transcript_pairwise.index)
print(len(core_metabolic_set_))

flux_ham_dists_selected = flux_ham_dist_df.loc[core_metabolic_set_, core_metabolic_set_]
transcript_pairwise_selected = transcript_pairwise.loc[core_metabolic_set_, core_metabolic_set_]

flux_ham_dist_df_essential_long = to_long_no_diagonal(flux_ham_dist_df_essential)
transcript_dist_selected_long = to_long_no_diagonal(transcript_pairwise_selected)

joint_dists = flux_ham_dist_df_essential_long.merge(
    transcript_dist_selected_long,
    on=["gene1", "gene2", "gene_pair"],
    suffixes=["_flux", "_transcript"],
)

In [ ]:
corr_ = stats.spearmanr(joint_dists["distance_flux"], joint_dists["distance_transcript"])[0]

fig = (
    gg.ggplot(joint_dists, gg.aes(x="distance_flux", y="distance_transcript"))
    + gg.geom_point(size=0.5)
    + gg.theme_minimal()
    + gg.labs(x="Flux distance", y="Transcript distance", title=f"Spearman rho = {corr_:.2f}")
    + SHARED_THEME
    + gg.theme(figure_size=(3, 2))
)
fig.save("joint_dists_flux_transcript.png", dpi=500, bbox_inches="tight")
display(fig)

In [ ]:
import pandas as pd
from scipy.spatial.distance import squareform
from scipy.cluster.hierarchy import linkage, fcluster

condensed_dist = squareform(flux_ham_dists_selected.values)
Z = linkage(condensed_dist, method="complete")
distance_threshold = 40.0
clusters_dist = fcluster(Z, t=distance_threshold, criterion="distance")
cluster_assignments = pd.Series(clusters_dist, index=flux_ham_dists_selected.index)

In [ ]:
# import pandas as pd
# from scipy.spatial.distance import squareform
# from scipy.cluster.hierarchy import linkage, fcluster

# condensed_dist = squareform(flux_ham_dists_selected.values)
# Z = linkage(condensed_dist, method="complete")
# num_clusters = 20  # Specify the target number of clusters
# clusters_dist = fcluster(Z, t=num_clusters, criterion="maxclust")
# cluster_assignments = pd.Series(clusters_dist, index=flux_ham_dists_selected.index)

In [ ]:
unique_clusters = sorted(cluster_assignments.unique())
palette = sns.color_palette("tab10", n_colors=len(unique_clusters))
cluster_color_map = dict(zip(unique_clusters, palette))

row_colors = cluster_assignments.map(cluster_color_map)

g = sns.clustermap(
    flux_ham_dists_selected,
    row_linkage=Z,
    col_linkage=Z,
    row_colors=row_colors,
    col_colors=row_colors,
    cmap="rocket_r",
    xticklabels=False,
    yticklabels=True,
    # vmax=20,
    cbar_pos=(0.02, 0.8, 0.05, 0.18),
    cbar_kws={"label": "Hamming Distance"},
)

legend_patches = [
    mpatches.Patch(color=color, label=f"Cluster {cluster_id}")
    for cluster_id, color in cluster_color_map.items()
]

plt.savefig("./flux_clustermap_core_metabolic_set.png", dpi=500, bbox_inches="tight")
plt.show()

In [ ]:
joint_dists.loc[lambda x: x["gene_pair"] == "lpxD_lpxK"]

In [ ]:
for cluster_id in range(1, len(cluster_assignments.unique() + 1)):
    gene_list = cluster_assignments[cluster_assignments == cluster_id].index.tolist()
    if len(gene_list) <= 1:
        continue
    print(",".join(gene_list))
    print()

    # flux_dist_selected = flux_ham_dist_df_essential.loc[gene_inter, gene_inter]
    # transcript_dist_selected = transcript_pairwise.loc[gene_inter, gene_inter]

    # g_flux = sns.clustermap(
    #     flux_dist_selected,
    #     cmap="rocket_r",
    #     xticklabels=False,
    #     yticklabels=True,
    #     vmax=np.quantile(flux_dist_selected.values, 0.9),
    # )
    # plt.show()

    # sns.clustermap(
    #     transcript_dist_selected,
    #     cmap="rocket_r",
    #     xticklabels=False,
    #     yticklabels=True,
    #     row_linkage=g_flux.dendrogram_row.linkage,
    #     col_linkage=g_flux.dendrogram_col.linkage,
    #     vmax=np.quantile(transcript_dist_selected.values, 0.9),
    # )
    # plt.show()
    joint_dists_ = joint_dists.loc[lambda x: x["gene1"].isin(gene_list)].loc[
        lambda x: x["gene2"].isin(gene_list)
    ]
    display(joint_dists_.sort_values("distance_transcript", ascending=False).head(20))

    fig = (
        gg.ggplot(
            joint_dists.loc[lambda x: x["gene1"].isin(gene_list)].loc[
                lambda x: x["gene2"].isin(gene_list)
            ],
            gg.aes(x="distance_flux", y="distance_transcript"),
        )
        + gg.geom_point()
        + gg.theme_minimal()
        + gg.labs(x="Flux distance", y="Transcript distance", title=",".join(gene_list))
        + SHARED_THEME
        + gg.theme(figure_size=(3, 2))
    )
    fig.save(f"joint_distances_cluster{cluster_id}.png", dpi=500, bbox_inches="tight")
    display(fig)

    # fig = px.scatter(
    #     joint_dists_,
    #     x="distance_flux",
    #     y="distance_transcript",
    #     hover_name="gene_pair",
    #     template="plotly_white",
    #     width=600,
    #     height=400,
    # )
    # fig.update_traces(marker=dict(size=3))
    # fig.show()

In [ ]:
import plotly.express as px


fig = px.scatter(
    joint_dists,
    x="distance_flux",
    y="distance_transcript",
    hover_name="gene_pair",
    template="plotly_white",
    width=600,
    height=400,
)
fig.update_traces(marker=dict(size=3))
fig.show()

In [ ]:
joint_dists.query("distance_flux < 20.0").sort_values("distance_transcript", ascending=False).head(
    20
)

In [ ]:
gene1 = "lptB"
gene2 = "lptF"

obs_subset = adata_case.obs.loc[adata_case.obs["gene"].isin([gene1, gene2])]
obs_subset["gene"] = obs_subset["gene"].astype(str)
(
    gg.ggplot(adata_case.obs, gg.aes(x="transcript_case_UMAP1", y="transcript_case_UMAP2"))
    + gg.geom_point(alpha=0.1)
    + gg.geom_point(
        obs_subset,
        gg.aes(x="transcript_case_UMAP1", y="transcript_case_UMAP2", color="gene"),
    )
    + gg.theme_minimal()
    + gg.labs(x="UMAP1", y="UMAP2")
    + gg.theme(figure_size=(3, 2))
)